Sinh viên thực hiện: Phạm Minh Phú - 2550187

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Bài 1: Titanic Problem - Linear Regression and Logistic Regression

## 1. Khai báo đường dẫn

In [2]:
from pathlib import Path

DATA_PATH = Path("Titanic_Dataset")

TRAIN_OUTPUT_PATH = DATA_PATH / "X_train_processed.csv"
VALIDATION_OUTPUT_PATH = DATA_PATH / "X_val_processed.csv"
TEST_OUTPUT_PATH = DATA_PATH / "X_test_processed.csv"

print("Đường dẫn:", DATA_PATH.resolve())
print("File tồn tại:", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file tại: {DATA_PATH.resolve()}"
    )

Đường dẫn: D:\minhphu\ML lab\python_ws\mliot-pyml-2026-hw\week04\Homework_b7\Titanic_Dataset
File tồn tại: True


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df_train = pd.read_csv(TRAIN_OUTPUT_PATH)
df_val = pd.read_csv(VALIDATION_OUTPUT_PATH)
df_test = pd.read_csv(TEST_OUTPUT_PATH)

print("Saved train shape:", df_train.shape)
print("Saved validation shape:", df_val.shape)
print("Saved test shape:", df_test.shape)

print("\nMissing trong train:", df_train.isna().sum().sum())
print("Missing trong validation:", df_val.isna().sum().sum())
print("Missing trong test:", df_test.isna().sum().sum())

print("\nCác cột trong train:")
print(df_train.columns.tolist())

Saved train shape: (623, 9)
Saved validation shape: (134, 9)
Saved test shape: (134, 9)

Missing trong train: 0
Missing trong validation: 0
Missing trong test: 0

Các cột trong train:
['num__age', 'num__sibsp', 'num__parch', 'num__fare', 'cat__sex_male', 'cat__embarked_Q', 'cat__embarked_S', 'ord__pclass', 'survived']


## 2. Tách target

In [4]:
X_train = df_train.copy().drop(columns=["survived"])
y_train = df_train.copy()["survived"]

X_val = df_val.copy().drop(columns=["survived"])
y_val = df_val.copy()["survived"]

X_test = df_test.copy().drop(columns=["survived"])
y_test = df_test.copy()["survived"]


## 3. LinearRegression

In [5]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_val)
print(y_pred.max(), y_pred.min(), y_pred.mean())
best_threshold = -0.2
best_f1 = f1_score(y_val, (y_pred >= best_threshold).astype(int))

t = best_threshold + 0.01
while t <= 1.1:
    f1 = f1_score(y_val, (y_pred >= t).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t
    t += 0.01

y_pred_test = (lr_model.predict(X_test) >= best_threshold).astype(int)

print("Đánh giá mô hình Linear Regression trên tập test:")
print("Accuracy:", accuracy_score(y_test, y_pred_test))
print("Precision:", precision_score(y_test, y_pred_test))
print("Recall:", recall_score(y_test, y_pred_test))
print("F1 Score:", f1_score(y_test, y_pred_test))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_test))


1.0504693254371198 -0.11776677997620211 0.42536381611259466
Đánh giá mô hình Linear Regression trên tập test:
Accuracy: 0.7761194029850746
Precision: 0.7692307692307693
Recall: 0.5882352941176471
F1 Score: 0.6666666666666666
Confusion Matrix:
 [[74  9]
 [21 30]]


## 4. LogisticRegression

In [6]:
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"],
    "fit_intercept": [True, False]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)
log_model = grid.best_estimator_

y_pred_proba = log_model.predict_proba(X_val)[:, 1]
print(y_pred_proba.max(), y_pred_proba.min(), y_pred_proba.mean())
best_threshold = 0
best_f1 = f1_score(y_val, (y_pred_proba >= best_threshold).astype(int))

t = best_threshold + 0.01
while t <= 1:
    f1 = f1_score(y_val, (y_pred_proba >= t).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t
    t += 0.01

y_pred = (log_model.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)
print("Đánh giá mô hình Logistic Regression trên tập test:")
print("Accuracy:", accuracy_score(y_test, y_pred.round(3)))
print("Precision:", precision_score(y_test, y_pred.round(3)))
print("Recall:", recall_score(y_test, y_pred.round(3)))
print("F1 Score:", f1_score(y_test, y_pred.round(3)))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred.round(3)))


0.946075467670064 0.08488220254731486 0.4200542692118718
Đánh giá mô hình Logistic Regression trên tập test:
Accuracy: 0.7835820895522388
Precision: 0.7894736842105263
Recall: 0.5882352941176471
F1 Score: 0.6741573033707865
Confusion Matrix:
 [[75  8]
 [21 30]]


# Trả lời bài 1

Dựa trên các chỉ số đánh giá, cả hai mô hình đều cho kết quả khá tương đồng, tuy nhiên **Logistic Regression** nhỉnh hơn **Linear Regression** ở hầu hết các tiêu chí. Cụ thể, **Logistic Regression** đạt *Accuracy* = 0.7836, cao hơn **Linear Regression** (0.7761), đồng thời có *Precision* = 0.7895 và *F1-score* = 0.6742, cũng cao hơn so với 0.7692 và 0.6667 của **Linear Regression**. Hai mô hình có *Recall* bằng nhau (0.5882), tức đều phát hiện được cùng số lượng hành khách sống sót. Ma trận nhầm lẫn cũng cho thấy **Logistic Regression** dự đoán đúng thêm một trường hợp không sống sót (75 so với 74) và giảm một trường hợp dự đoán sai (8 so với 9).

Vì đây là bài toán phân loại nhị phân, **Logistic Regression** là lựa chọn phù hợp hơn. Mô hình này được thiết kế chuyên cho bài toán phân loại, dự đoán trực tiếp xác suất thuộc từng lớp và cho hiệu quả tốt hơn **Linear Regression**, vốn chủ yếu được xây dựng cho bài toán hồi quy và phải chuyển đổi kết quả liên tục thành nhãn phân loại bằng cách đặt ngưỡng. Do đó, **Logistic Regression** vừa có cơ sở lý thuyết phù hợp hơn, vừa đạt kết quả thực nghiệm tốt hơn trên tập dữ liệu *Titanic*.

## Bài 2: Dry Bean Problem - Logistic Regression and KNN

## 1. Khai báo đường dẫn

In [7]:
from pathlib import Path

DATA_PATH = Path("Dry_Bean_Dataset")

TRAIN_OUTPUT_PATH = DATA_PATH / "dry_bean_train.csv"
TEST_OUTPUT_PATH = DATA_PATH / "dry_bean_test.csv"

print("Đường dẫn:", DATA_PATH.resolve())
print("File tồn tại:", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file tại: {DATA_PATH.resolve()}"
    )

Đường dẫn: D:\minhphu\ML lab\python_ws\mliot-pyml-2026-hw\week04\Homework_b7\Dry_Bean_Dataset
File tồn tại: True


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df_train = pd.read_csv(TRAIN_OUTPUT_PATH)
df_test = pd.read_csv(TEST_OUTPUT_PATH)

print("Saved train shape:", df_train.shape)
print("Saved test shape:", df_test.shape)

print("\nMissing trong train:", df_train.isna().sum().sum())
print("Missing trong test:", df_test.isna().sum().sum())

print("\nCác cột trong train:")
print(df_train.columns.tolist())

print(df_train.head())
print(df_test.head())

Saved train shape: (10834, 17)
Saved test shape: (2709, 17)

Missing trong train: 0
Missing trong test: 0

Các cột trong train:
['area', 'perimeter', 'majoraxislength', 'minoraxislength', 'aspectration', 'eccentricity', 'convexarea', 'equivdiameter', 'extent', 'solidity', 'roundness', 'compactness', 'shapefactor1', 'shapefactor2', 'shapefactor3', 'shapefactor4', 'class']
    area  perimeter  majoraxislength  minoraxislength  aspectration  \
0  69471   1069.638       399.100245       225.005782      1.773733   
1  82877   1162.581       391.817013       270.836144      1.446694   
2  65042   1023.506       419.202858       198.962774      2.106941   
3  41315    758.920       287.438268       183.447580      1.566869   
4  91088   1168.645       459.300729       253.950486      1.808623   

   eccentricity  convexarea  equivdiameter    extent  solidity  roundness  \
0      0.825923       71088     297.410868  0.707386  0.977254   0.763027   
1      0.722634       84171     324.841921  0

## 2. Tách target và tiền xử lý

In [9]:
from sklearn.preprocessing import RobustScaler, LabelEncoder


X_train = df_train.copy().drop(columns=["class"])
y_train = df_train.copy()["class"]

X_test = df_test.copy().drop(columns=["class"])
y_test = df_test.copy()["class"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), X_train.columns)
    ]
)

X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)
print("Shapes sau biến đổi:", X_train_scaled.shape, X_test_scaled.shape)
print("Tên cột:", list(preprocessor.get_feature_names_out()))

label_encoder = LabelEncoder()
y_train_scaled = label_encoder.fit_transform(y_train)
y_test_scaled = label_encoder.transform(y_test)

print(X_train_scaled[:5,:], y_train_scaled[:5])
print(X_test_scaled[:5,:], y_test_scaled[:5])


Shapes sau biến đổi: (10834, 16) (2709, 16)
Tên cột: ['num__area', 'num__perimeter', 'num__majoraxislength', 'num__minoraxislength', 'num__aspectration', 'num__eccentricity', 'num__convexarea', 'num__equivdiameter', 'num__extent', 'num__solidity', 'num__roundness', 'num__compactness', 'num__shapefactor1', 'num__shapefactor2', 'num__shapefactor3', 'num__shapefactor4']
[[ 0.98858915  1.01022377  0.83510746  0.78786633  0.82729832  0.66245874
   1.00923442  0.91265284 -0.76707898 -2.53971363 -1.4390932  -0.79464168
  -0.65361301 -0.597045   -0.76936902 -2.73457654]
 [ 1.52159911  1.35071579  0.77581284  1.89985495 -0.37982681 -0.44084923
   1.51792449  1.33629784  0.97618783 -0.84175826 -1.34929283  0.38752763
  -1.3951342  -0.31612734  0.3955452  -0.47821435]
 [ 0.81249627  0.8412215   0.99876784  0.15598112  2.05719196  1.24211909
   0.80160582  0.76382571  0.35026077  0.22576895 -1.23357784 -1.62244551
  -0.1431098  -0.80399639 -1.51120815 -0.83390047]
 [-0.13086695 -0.12807604 -0.0739

## 3. Logistic Regression

In [10]:
log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train_scaled)

y_pred = log_model.predict(X_test_scaled)
print("Đánh giá mô hình Logistic Regression trên tập test:")
print("Accuracy:", accuracy_score(y_test_scaled, y_pred))
print("Precision:", precision_score(y_test_scaled, y_pred, average='macro'))
print("Recall:", recall_score(y_test_scaled, y_pred, average='macro'))
print("F1 Score:", f1_score(y_test_scaled, y_pred, average='macro'))
print("Confusion Matrix:\n", confusion_matrix(y_test_scaled, y_pred))


Đánh giá mô hình Logistic Regression trên tập test:
Accuracy: 0.9187892211148025
Precision: 0.9303870005069063
Recall: 0.9299652263267638
F1 Score: 0.930056806183272
Confusion Matrix:
 [[237   0  17   0   0   4   7]
 [  0 104   0   0   0   0   0]
 [  8   0 307   0   5   2   4]
 [  0   0   0 643   0  13  53]
 [  1   0  11   5 351   0   4]
 [ 10   0   0   5   0 382   9]
 [  1   0   1  42   8  10 465]]


## 4. KNN

In [12]:
knn_model = KNeighborsClassifier(n_neighbors=9)
knn_model.fit(X_train_scaled, y_train_scaled)

y_pred = knn_model.predict(X_test_scaled)
print("Đánh giá mô hình Logistic Regression trên tập test:")
print("Accuracy:", accuracy_score(y_test_scaled, y_pred))
print("Precision:", precision_score(y_test_scaled, y_pred, average='macro'))
print("Recall:", recall_score(y_test_scaled, y_pred, average='macro'))
print("F1 Score:", f1_score(y_test_scaled, y_pred, average='macro'))
print("Confusion Matrix:\n", confusion_matrix(y_test_scaled, y_pred))

Đánh giá mô hình Logistic Regression trên tập test:
Accuracy: 0.9147286821705426
Precision: 0.9293852239334843
Recall: 0.925316200940992
F1 Score: 0.9270777753000132
Confusion Matrix:
 [[233   0  20   0   1   4   7]
 [  0 104   0   0   0   0   0]
 [  7   0 308   0   6   2   3]
 [  0   0   0 647   0  11  51]
 [  0   0  14   5 346   0   7]
 [  5   0   0   6   0 383  12]
 [  3   0   0  55   6   6 457]]
